In [29]:
#Create the main Repo list
import os
import pandas as pd
from urllib.parse import urlparse

# === INPUT / OUTPUT ===
SRC_CSV = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Clone_Status.csv"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
OUT_CSV = os.path.join(OUT_DIR, "3.2_Total_Repo.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# === Load ===
df = pd.read_csv(SRC_CSV, dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]

# --- Find columns ---
def pick_col(candidates, cols):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

clone_col = pick_col(["clone_status"], df.columns)
yml_col   = pick_col(["yml_detected"], df.columns)
url_col   = pick_col(["html_url"], df.columns)

if not clone_col or not yml_col or not url_col:
    missing = [name for name, col in {"clone_status": clone_col, "yml_detected": yml_col, "html_url/htm_url": url_col}.items() if not col]
    raise ValueError(f"Missing required column(s): {', '.join(missing)}")

# --- Normalize "yes" detection ---
def is_yes(x: str) -> bool:
    return str(x).strip().lower() in {"yes", "true", "y", "1"}

filtered = df[ df[clone_col].apply(is_yes) & df[yml_col].apply(is_yes) ].copy()

# --- Build full_name = owner.repo ---
def url_to_full_name(u: str) -> str:
    try:
        path = urlparse(str(u).strip()).path.strip("/")
        if not path:
            return ""
        if path.endswith(".git"):
            path = path[:-4]
        parts = path.split("/")
        if len(parts) >= 2:
            return f"{parts[0]}.{parts[1]}"
        return path
    except Exception:
        return ""

# Ensure full_name is lowercase
filtered["full_name"] = filtered[url_col].apply(url_to_full_name).str.lower()

# --- Keep only URL + full_name ---
out = filtered[[url_col, "full_name"]].rename(columns={url_col: "html_url"})

# Drop duplicates
out = out.drop_duplicates(subset=["html_url"]).reset_index(drop=True)

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518)


In [30]:
#Adds the Instru_tests for each repo

import os
import pandas as pd
from collections import defaultdict

# === PATHS ===
REPO_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv"
TEST_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Test_Files"
OUT_CSV  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv"

# === Load repo list ===
repos = pd.read_csv(REPO_CSV, dtype=str).fillna("")
if "full_name" not in repos.columns:
    raise ValueError("Expected column 'full_name' in 3.1_Total_Repo.csv (format: owner.repo).")
repos["full_name"] = repos["full_name"].astype(str).str.strip()

# === Detection keywords ===
INSTRU_HINTS_NATIVE  = [
    "instrumentation", "androidtest", "connectedandroidtest",
    "espresso", "uiautomator", "orchestrator", "manageddevices", "gmd"
]
INSTRU_HINTS_FLUTTER = [
    "flutter", "dart"
]

def is_instru_file(fname: str) -> bool:
    name = fname.lower()
    return any(k in name for k in INSTRU_HINTS_NATIVE + INSTRU_HINTS_FLUTTER)

def classify_test(fname: str) -> str:
    lname = fname.lower()
    if any(k in lname for k in INSTRU_HINTS_FLUTTER):
        return "flutter"
    if any(k in lname for k in INSTRU_HINTS_NATIVE):
        return "native"
    return ""

# === Count tests per repo ===
native_counts  = defaultdict(int)
flutter_counts = defaultdict(int)

for fname in os.listdir(TEST_DIR):
    fpath = os.path.join(TEST_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    if "__" not in fname:
        continue

    repo_token = fname.split("__", 1)[0].strip()
    if not repo_token:
        continue
    if not is_instru_file(fname):
        continue

    kind = classify_test(fname)
    if kind == "flutter":
        flutter_counts[repo_token] += 1
    elif kind == "native":
        native_counts[repo_token] += 1

# === Merge into repo DataFrame ===
repos["native_instru_test"]  = repos["full_name"].map(lambda k: native_counts.get(k, 0)).astype(int)
repos["flutter_instru_test"] = repos["full_name"].map(lambda k: flutter_counts.get(k, 0)).astype(int)
repos["Intru_test"] = (repos["native_instru_test"] + repos["flutter_instru_test"] > 0)

# === Save ===
repos.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(repos)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv (rows=4518)


In [31]:
# Aggregate the yml files' instru test analysis for each repo in the main list

import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
YML_CSV  = os.path.join(BASE_DIR, "3.1_YML_Files.csv")
OUT_CSV  = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")

def unique_preserve(items):
    seen = set()
    out = []
    for x in items:
        x = str(x).strip()
        if not x:
            continue
        parts = [p.strip() for p in x.split(",") if p.strip()]
        for p in parts:
            if p not in seen:
                seen.add(p)
                out.append(p)
    return out

# --- Load data ---
main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
yml  = pd.read_csv(YML_CSV, dtype=str).fillna("")

# sanity checks
required_yml_cols = {
    "full_name", "ci_platform",
    "has_device_setup", "has_test_trigger",
    "device_setup_group", "test_trigger_group"
}
missing = required_yml_cols - set(map(str, yml.columns))
if missing:
    raise ValueError(f"3.1_YML_Files.csv is missing columns: {', '.join(sorted(missing))}")

if "full_name" not in main.columns:
    raise ValueError("3.4_Total_Repo.csv must contain 'full_name'.")

# normalize booleans
def to_bool(s):
    return str(s).strip().lower() in {"true", "yes", "y", "1"}

yml["has_device_setup"] = yml["has_device_setup"].map(to_bool)
yml["has_test_trigger"] = yml["has_test_trigger"].map(to_bool)
#yml["fallback_detected"] = yml["fallback_detected"].map(to_bool)

# --- Aggregate per repo ---
if not yml.empty:
    plat_counts = (
        yml.groupby(["full_name", "ci_platform"])
           .size()
           .unstack(fill_value=0)
    )
    total_counts = yml.groupby("full_name").size().rename("num_yml_files")

    bool_agg = (
        yml.groupby("full_name")[["has_device_setup", "has_test_trigger"]]
           .any()
           .reset_index()
    )

    group_agg = (
        yml.groupby("full_name")[["device_setup_group", "test_trigger_group"]]
           .agg(lambda col: ", ".join(unique_preserve(col.tolist())))
           .reset_index()
    )

    plat_list = (
        yml.groupby("full_name")["ci_platform"]
           .agg(lambda col: ", ".join(unique_preserve(col.tolist())))
           .rename("ci_platform")
           .reset_index()
    )

    agg = bool_agg.merge(group_agg, on="full_name", how="left") \
                  .merge(plat_list, on="full_name", how="left") \
                  .merge(total_counts, on="full_name", how="left")

    if not plat_counts.empty:
        plat_counts = plat_counts.add_prefix("num_yml_").reset_index()
        agg = agg.merge(plat_counts, on="full_name", how="left")

    count_cols = [c for c in agg.columns if c.startswith("num_yml_")]
    agg[count_cols] = agg[count_cols].fillna(0).astype(int)

else:
    agg = pd.DataFrame(columns=[
        "full_name", "has_device_setup", "has_test_trigger","fallback_detected"
        "device_setup_group", "test_trigger_group",
        "ci_platform", "num_yml_files"
    ])

# --- Left join onto main ---
out = main.merge(agg, on="full_name", how="left")

# fill defaults
out["has_device_setup"] = out["has_device_setup"].fillna(False).astype(bool)
out["has_test_trigger"] = out["has_test_trigger"].fillna(False).astype(bool)
#out["fallback_detected"] = out["fallback_detected"].fillna(False).astype(bool)
for col in ["device_setup_group", "test_trigger_group", "ci_platform"]:
    if col in out.columns:
        out[col] = out[col].fillna("")
if "num_yml_files" in out.columns:
    out["num_yml_files"] = out["num_yml_files"].fillna(0).astype(int)
for col in out.columns:
    if col.startswith("num_yml_"):
        out[col] = out[col].fillna(0).astype(int)

# --- Add instru_t_ci ---
out["instru_t_ci"] = out["has_device_setup"] | out["has_test_trigger"]

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  (rows={len(out)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv  (rows=4518)


In [32]:
# update the device_setup_group and test_trigger_group to make their value sorted and consistent

# -*- coding: utf-8 -*-
import os
import re
import pandas as pd

BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
INPUT_CSV  = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")

def clean_cell(val: str) -> str:
    """Split by commas, strip whitespace, drop empties, de-dupe, sort, and rejoin."""
    if pd.isna(val) or str(val).strip() == "":
        return ""
    # split on commas with optional whitespace around them
    parts = [p.strip() for p in re.split(r"\s*,\s*", str(val)) if p.strip()]
    # de-dupe (exact-match) then sort case-insensitively but keep original casing
    unique_sorted = sorted(set(parts), key=lambda s: s.casefold())
    return ", ".join(unique_sorted)

def main():
    if not os.path.isfile(INPUT_CSV):
        raise FileNotFoundError(f"Input not found: {INPUT_CSV}")

    df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")

    # Ensure columns exist
    for col in ("device_setup_group", "test_trigger_group"):
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in {INPUT_CSV}")

    # Clean columns
    df["device_setup_group"] = df["device_setup_group"].apply(clean_cell)
    df["test_trigger_group"] = df["test_trigger_group"].apply(clean_cell)

    # Save
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved cleaned CSV -> {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


Saved cleaned CSV -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv
